# Graph Neural Networks: From Scratch

**What we'll build:** A complete Graph Neural Network (GNN) from scratch that learns to classify nodes in a social network.

**Why it matters:** Many real-world systems are naturally graphs — social networks, molecules, knowledge graphs, citation networks. Traditional neural networks struggle with graph-structured data because:
1. Graphs have **variable size** neighborhoods
2. There's **no fixed ordering** of neighbors
3. We need to reason about **relationships** between entities

**Intuitions we'll develop:**
- How graphs differ from images/sequences
- The core idea: **message passing** between neighbors
- How to aggregate information across variable-sized neighborhoods
- Why GNNs learn better representations than treating nodes independently

## 1. Setup

We'll use PyTorch for our implementation and matplotlib for visualization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.manifold import TSNE

from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

Set random seeds for reproducibility.

In [ ]:
set_seed(42)

Configure device (GPU if available).

In [ ]:
device = get_device()
print(f"Using device: {device}")

## 2. Graphs 101: The Basics

A **graph** $G = (V, E)$ consists of:
- **Nodes** (vertices) $V$: entities in the graph
- **Edges** $E$: connections between nodes

Let's start with the simplest possible graph: 4 friends in a social network.

In [ ]:
# Simple graph: 4 nodes (people), edges (friendships)
# Edge list: (person A, person B) means they're friends
edges = [
    (0, 1),  # Person 0 and 1 are friends
    (0, 2),  # Person 0 and 2 are friends
    (1, 2),  # Person 1 and 2 are friends
    (2, 3),  # Person 2 and 3 are friends
]

num_nodes = 4
print(f"Graph with {num_nodes} nodes and {len(edges)} edges")

Let's visualize this simple social network.

In [ ]:
# Create networkx graph for visualization
G = nx.Graph()
G.add_nodes_from(range(num_nodes))
G.add_edges_from(edges)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', 
        node_size=1000, font_size=16, font_weight='bold',
        edge_color='gray', width=2)
plt.title('Simple Social Network', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

**Key observation:** Person 2 is the most connected (has 3 friends), while Person 3 has only 1 friend. The structure matters!

### Adjacency Matrix Representation

The **adjacency matrix** $A$ is a square matrix where $A_{ij} = 1$ if there's an edge from node $i$ to node $j$, otherwise $0$.

This is the most common representation for GNNs.

In [ ]:
# Create adjacency matrix from edge list
def create_adjacency_matrix(num_nodes, edges):
    A = torch.zeros(num_nodes, num_nodes)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1  # Undirected graph
    return A

A = create_adjacency_matrix(num_nodes, edges)
print("Adjacency Matrix:")
print(A.numpy().astype(int))

Visualize the adjacency matrix as a heatmap.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(A.numpy(), cmap='Blues', interpolation='nearest')
plt.colorbar(label='Connection')
plt.xlabel('Node')
plt.ylabel('Node')
plt.title('Adjacency Matrix Visualization')
plt.xticks(range(num_nodes))
plt.yticks(range(num_nodes))

# Add values
for i in range(num_nodes):
    for j in range(num_nodes):
        plt.text(j, i, int(A[i, j].item()), 
                ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

**Key insight:** The adjacency matrix is symmetric (since our graph is undirected) and sparse (most entries are 0 in real graphs).

## 3. Node Features: Representing Information

Each node has **features** — properties that describe it. In a social network, this could be age, interests, location, etc.

Let's give each person a simple 2D feature vector (think: "outdoorsy" vs "tech-savvy" scores).

In [ ]:
# Node features: [outdoorsy_score, tech_score]
X = torch.tensor([
    [1.0, 0.2],  # Person 0: loves outdoors, low tech
    [0.8, 0.4],  # Person 1: somewhat outdoorsy, medium tech
    [0.3, 0.9],  # Person 2: indoor person, high tech
    [0.2, 0.8],  # Person 3: indoor person, high tech
], dtype=torch.float32)

print("Node Features (shape:", X.shape, ")")
print(X)

Visualize the features in 2D space.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=300, c=range(num_nodes), cmap='viridis', edgecolors='black', linewidth=2)

for i in range(num_nodes):
    plt.annotate(f'Person {i}', (X[i, 0], X[i, 1]), 
                xytext=(10, 10), textcoords='offset points', fontsize=12)

plt.xlabel('Outdoorsy Score', fontsize=12)
plt.ylabel('Tech Score', fontsize=12)
plt.title('Node Features in 2D Space', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** Persons 2 and 3 are similar (both high tech), but we haven't used the graph structure yet!

## 4. The Problem: Why Traditional NNs Fail

What if we tried to use a simple fully-connected neural network on each node independently?

In [ ]:
# Simple MLP applied to each node independently
class IndependentMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, hidden_dim)
    
    def forward(self, x):
        return torch.relu(self.linear(x))

mlp = IndependentMLP(input_dim=2, hidden_dim=4)
mlp_output = mlp(X)

print("MLP Output (each node processed independently):")
print(mlp_output)

**The critical flaw:** This MLP treats each node in isolation — it never looks at neighbors!

In a social network:
- Person 0's interests might be influenced by their friends
- Person 2's popularity (3 connections) is invisible to the model
- The entire graph structure is ignored

**We need a way to incorporate neighborhood information!**

## 5. The Core Idea: Message Passing

**Graph Neural Networks** solve this with **message passing**:

1. Each node receives "messages" from its neighbors
2. These messages are aggregated (e.g., summed or averaged)
3. The node updates its representation based on aggregated messages

Mathematically, for node $i$ at layer $l+1$:

$$h_i^{(l+1)} = \sigma\left(W^{(l)} \cdot \text{AGGREGATE}\left(\{h_j^{(l)} : j \in \mathcal{N}(i)\}\right)\right)$$

Where:
- $h_i^{(l)}$ is the hidden representation of node $i$ at layer $l$
- $\mathcal{N}(i)$ is the set of neighbors of node $i$
- $W^{(l)}$ is a learnable weight matrix
- $\sigma$ is a non-linear activation function
- AGGREGATE is a function like SUM, MEAN, or MAX

### Simplest Message Passing: Sum Neighbors

Let's start with the absolute simplest version: just sum up neighbor features (no weights yet).

In [ ]:
def simple_message_passing(X, A):
    """
    Simple message passing: sum neighbor features
    X: node features [num_nodes, feature_dim]
    A: adjacency matrix [num_nodes, num_nodes]
    """
    # Matrix multiplication A @ X gives sum of neighbor features!
    return A @ X

# Aggregate neighbor features
aggregated = simple_message_passing(X, A)

print("Original Features:")
print(X)
print("\nAggregated Neighbor Features:")
print(aggregated)

Let's understand what happened for Person 2 (who has 3 neighbors).

In [ ]:
person_2_neighbors = [0, 1, 3]  # From our edge list

print("Person 2's neighbors:")
for neighbor in person_2_neighbors:
    print(f"  Person {neighbor}: {X[neighbor].tolist()}")

print(f"\nSum of neighbor features: {X[person_2_neighbors].sum(dim=0).tolist()}")
print(f"Aggregated result for Person 2: {aggregated[2].tolist()}")
print("\nThey match! This is message passing in action.")

**Key insight:** Matrix multiplication `A @ X` elegantly computes the sum of neighbor features for all nodes in parallel!

But there's a problem: nodes with more neighbors get larger values. Let's normalize.

### Normalized Message Passing

We should average neighbor features (not sum) to avoid bias toward high-degree nodes.

In [ ]:
def normalized_message_passing(X, A):
    """
    Normalized message passing: average neighbor features
    """
    # Compute degree of each node (number of neighbors)
    degree = A.sum(dim=1, keepdim=True)  # [num_nodes, 1]
    degree[degree == 0] = 1  # Avoid division by zero
    
    # Normalize: divide by degree
    A_normalized = A / degree
    
    return A_normalized @ X

# Aggregate and normalize
aggregated_norm = normalized_message_passing(X, A)

print("Normalized Aggregated Features:")
print(aggregated_norm)
print(f"\nPerson 2 (3 neighbors): {aggregated_norm[2].tolist()}")
print(f"Person 3 (1 neighbor): {aggregated_norm[3].tolist()}")

**Much better!** Now the magnitudes are comparable across nodes with different degrees.

## 6. Building a GNN Layer

Now let's add learnable parameters and create a proper GNN layer.

A **Graph Convolutional Network (GCN)** layer does:
1. Aggregate neighbor features (with normalization)
2. Apply a linear transformation (learnable weights)
3. Add non-linearity

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, X, A):
        """
        X: node features [num_nodes, input_dim]
        A: adjacency matrix [num_nodes, num_nodes]
        """
        # Step 1: Normalize adjacency matrix
        degree = A.sum(dim=1, keepdim=True)
        degree[degree == 0] = 1
        A_norm = A / degree
        
        # Step 2: Aggregate neighbor features
        aggregated = A_norm @ X
        
        # Step 3: Apply linear transformation
        out = self.linear(aggregated)
        
        return out

# Test the layer
gcn_layer = GCNLayer(input_dim=2, output_dim=4)
output = gcn_layer(X, A)

print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"\nOutput features:")
print(output)

**What's happening:** Each node now has a 4-dimensional representation that incorporates information from its neighbors!

### Adding Self-Loops

One issue: the node only looks at neighbors, not itself! We should add **self-loops** — edges from each node to itself.

In [ ]:
def add_self_loops(A):
    """
    Add self-loops to adjacency matrix (add identity matrix)
    """
    num_nodes = A.shape[0]
    return A + torch.eye(num_nodes)

A_with_loops = add_self_loops(A)

print("Original Adjacency Matrix:")
print(A.numpy().astype(int))
print("\nWith Self-Loops (diagonal = 1):")
print(A_with_loops.numpy().astype(int))

Update our GCN layer to include self-loops by default.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, X, A):
        # Add self-loops
        A_hat = A + torch.eye(A.shape[0]).to(A.device)
        
        # Normalize
        degree = A_hat.sum(dim=1, keepdim=True)
        degree[degree == 0] = 1
        A_norm = A_hat / degree
        
        # Aggregate and transform
        aggregated = A_norm @ X
        out = self.linear(aggregated)
        
        return out

# Test updated layer
gcn_layer = GCNLayer(input_dim=2, output_dim=4)
output_with_loops = gcn_layer(X, A)

print(f"Output with self-loops:")
print(output_with_loops)

**Why self-loops matter:** Without them, a node's own features are diluted. With them, the node retains its identity while incorporating neighborhood info.

## 7. Full GNN Model: Stacking Layers

Just like CNNs, we can stack multiple GNN layers. Each layer allows information to propagate further across the graph.

- **1 layer:** Each node sees its 1-hop neighbors
- **2 layers:** Each node sees its 2-hop neighbors
- **k layers:** Each node sees its k-hop neighbors

In [ ]:
class GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2):
        super().__init__()
        
        # First layer
        self.layers = nn.ModuleList([GCNLayer(input_dim, hidden_dim)])
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.layers.append(GCNLayer(hidden_dim, hidden_dim))
        
        # Output layer
        self.layers.append(GCNLayer(hidden_dim, output_dim))
    
    def forward(self, X, A):
        # Pass through all layers with ReLU activation
        for i, layer in enumerate(self.layers):
            X = layer(X, A)
            if i < len(self.layers) - 1:  # No activation after last layer
                X = torch.relu(X)
        return X

# Create a 2-layer GNN
gnn = GNN(input_dim=2, hidden_dim=8, output_dim=4, num_layers=2)
gnn_output = gnn(X, A)

print(f"Input shape: {X.shape}")
print(f"GNN output shape: {gnn_output.shape}")
print(f"\nNumber of parameters: {sum(p.numel() for p in gnn.parameters())}")

Visualize how information flows through the layers.

In [ ]:
# Track activations through layers
activations = [X]
x_temp = X

for i, layer in enumerate(gnn.layers):
    x_temp = layer(x_temp, A)
    if i < len(gnn.layers) - 1:
        x_temp = torch.relu(x_temp)
    activations.append(x_temp.detach())

# Plot magnitude of features at each layer
fig, axes = plt.subplots(1, len(activations), figsize=(15, 3))

for i, (ax, act) in enumerate(zip(axes, activations)):
    magnitudes = act.norm(dim=1).numpy()
    ax.bar(range(len(magnitudes)), magnitudes, color='steelblue')
    ax.set_title(f'Layer {i}' if i > 0 else 'Input', fontsize=12)
    ax.set_xlabel('Node', fontsize=10)
    ax.set_ylabel('Feature Magnitude', fontsize=10)
    ax.set_xticks(range(num_nodes))

plt.tight_layout()
plt.show()

**Observation:** Features get transformed and mixed through the layers, incorporating more neighborhood information at each step.

## 8. Real Example: Zachary's Karate Club

Let's test our GNN on a famous dataset: **Zachary's Karate Club**.

This is a social network of 34 members of a karate club. Due to a dispute, the club split into 2 groups. Can our GNN learn to predict which group each member joined?

**Task:** Node classification (predict community membership)

In [ ]:
# Load Zachary's Karate Club
karate = nx.karate_club_graph()

# Extract adjacency matrix
A_karate = torch.tensor(nx.to_numpy_array(karate), dtype=torch.float32)
num_nodes_karate = A_karate.shape[0]

# Extract labels (which community each person joined)
# 0 = Mr. Hi's group, 1 = Officer's group
labels = torch.tensor([karate.nodes[i]['club'] == 'Officer' 
                      for i in range(num_nodes_karate)], dtype=torch.long)

print(f"Graph: {num_nodes_karate} nodes, {karate.number_of_edges()} edges")
print(f"Community 0: {(labels == 0).sum()} members")
print(f"Community 1: {(labels == 1).sum()} members")

Visualize the karate club network colored by community membership.

In [ ]:
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(karate, seed=42)

# Color nodes by their community
colors = ['lightblue' if labels[i] == 0 else 'lightcoral' for i in range(num_nodes_karate)]

nx.draw(karate, pos, node_color=colors, with_labels=True, 
        node_size=500, font_size=10, font_weight='bold',
        edge_color='gray', width=1, alpha=0.7)

plt.title("Zachary's Karate Club\n(Blue = Mr. Hi's group, Red = Officer's group)", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

**Key insight:** Notice how nodes in the same community tend to cluster together. The graph structure contains information about community membership!

### Node Features: Degree as a Simple Feature

We don't have rich features for each person, so let's use a simple feature: the **degree** (number of connections) of each node.

In practice, you might use attributes like age, interests, etc., but degree is a good starting point.

In [ ]:
# Create simple node features: one-hot encoding of node identity + degree
# Using identity helps the model learn node-specific patterns
identity = torch.eye(num_nodes_karate)
degree = A_karate.sum(dim=1, keepdim=True)  # Degree as feature
X_karate = torch.cat([identity, degree / degree.max()], dim=1)  # Normalize degree

print(f"Node features shape: {X_karate.shape}")
print(f"Features: {num_nodes_karate} (identity) + 1 (degree) = {X_karate.shape[1]} dimensions")

## 9. Training the GNN

Let's train a GNN to predict community membership.

**Setup:**
- **Semi-supervised learning:** We'll use only a few labeled nodes for training
- **Training mask:** Randomly select 10 nodes for training
- **Validation:** Test on all remaining nodes

This mimics real-world scenarios where labels are expensive.

In [ ]:
# Create train/val split
num_train = 10
train_mask = torch.zeros(num_nodes_karate, dtype=torch.bool)
train_indices = torch.randperm(num_nodes_karate)[:num_train]
train_mask[train_indices] = True
val_mask = ~train_mask

print(f"Training nodes: {train_mask.sum()}")
print(f"Validation nodes: {val_mask.sum()}")

Create our GNN model for binary classification.

In [ ]:
# Initialize model
input_dim = X_karate.shape[1]
hidden_dim = 16
output_dim = 2  # Binary classification

model = GNN(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=2)
model = model.to(device)

# Move data to device
X_karate = X_karate.to(device)
A_karate = A_karate.to(device)
labels = labels.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

Define training loop with loss and accuracy tracking.

In [ ]:
def train_epoch(model, X, A, labels, train_mask, optimizer):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    logits = model(X, A)
    
    # Compute loss only on training nodes
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    return loss.item()

def evaluate(model, X, A, labels, mask):
    model.eval()
    with torch.no_grad():
        logits = model(X, A)
        preds = logits.argmax(dim=1)
        
        # Accuracy on masked nodes
        correct = (preds[mask] == labels[mask]).sum().item()
        accuracy = correct / mask.sum().item()
        
    return accuracy

Train the model and track metrics.

In [ ]:
# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
num_epochs = 200

train_losses = []
train_accs = []
val_accs = []

# Training loop
for epoch in range(num_epochs):
    loss = train_epoch(model, X_karate, A_karate, labels, train_mask, optimizer)
    train_acc = evaluate(model, X_karate, A_karate, labels, train_mask)
    val_acc = evaluate(model, X_karate, A_karate, labels, val_mask)
    
    train_losses.append(loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

print(f"\nFinal Validation Accuracy: {val_accs[-1]:.4f}")

Visualize training progress.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_losses, color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(train_accs, label='Train', color='green', linewidth=2)
axes[1].plot(val_accs, label='Validation', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Accuracy Over Time', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Great!** The model learns to classify nodes with high accuracy, even though we only labeled 10 out of 34 nodes for training.

## 10. Visualizing Learned Embeddings

Let's extract the learned node embeddings (from the hidden layer) and visualize them in 2D using t-SNE.

**Question:** Do nodes from the same community cluster together in the learned representation?

In [ ]:
# Extract embeddings from the first layer
model.eval()
with torch.no_grad():
    # Get output after first layer
    x = X_karate
    x = model.layers[0](x, A_karate)
    x = torch.relu(x)
    embeddings = x.cpu().numpy()

# Reduce to 2D using t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(embeddings)

print(f"Embeddings shape: {embeddings.shape}")
print(f"2D embeddings shape: {embeddings_2d.shape}")

Plot the learned embeddings colored by true community membership.

In [ ]:
plt.figure(figsize=(10, 8))

# Color by true labels
colors_labels = ['lightblue' if labels[i] == 0 else 'lightcoral' for i in range(num_nodes_karate)]

plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
           c=colors_labels, s=300, edgecolors='black', linewidth=2, alpha=0.7)

# Add node labels
for i in range(num_nodes_karate):
    plt.annotate(str(i), (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                ha='center', va='center', fontsize=9, fontweight='bold')

plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.title('Learned Node Embeddings (GNN)\n(Blue = Community 0, Red = Community 1)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Amazing!** The GNN learned to embed nodes such that members of the same community are close together in the embedding space.

This happened purely from the graph structure and a few training labels — the model discovered the community structure!

## 11. Comparison: GNN vs. No Graph Structure

Let's see what happens if we train a simple MLP that ignores the graph structure entirely.

In [ ]:
# Simple MLP (no graph structure)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, X):
        x = torch.relu(self.fc1(X))
        x = self.fc2(x)
        return x

# Train MLP
mlp = MLP(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim).to(device)
optimizer_mlp = torch.optim.Adam(mlp.parameters(), lr=0.01, weight_decay=5e-4)

mlp_val_accs = []

for epoch in range(num_epochs):
    mlp.train()
    optimizer_mlp.zero_grad()
    logits = mlp(X_karate)
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    loss.backward()
    optimizer_mlp.step()
    
    # Evaluate
    mlp.eval()
    with torch.no_grad():
        logits = mlp(X_karate)
        preds = logits.argmax(dim=1)
        val_acc = (preds[val_mask] == labels[val_mask]).float().mean().item()
        mlp_val_accs.append(val_acc)

print(f"MLP Final Validation Accuracy: {mlp_val_accs[-1]:.4f}")
print(f"GNN Final Validation Accuracy: {val_accs[-1]:.4f}")
print(f"\nImprovement with GNN: {(val_accs[-1] - mlp_val_accs[-1]) * 100:.1f}%")

Compare validation accuracy curves.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(val_accs, label='GNN', color='steelblue', linewidth=2.5)
plt.plot(mlp_val_accs, label='MLP (no graph)', color='coral', linewidth=2.5, linestyle='--')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('GNN vs MLP: Using Graph Structure Matters!', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Key insight:** The GNN significantly outperforms the MLP because it leverages the graph structure!

The MLP treats each node independently, while the GNN incorporates neighborhood information through message passing.

## 12. Understanding Multi-Layer Propagation

Let's visualize how information propagates through multiple GNN layers.

With **k layers**, each node aggregates information from its **k-hop neighborhood**.

In [ ]:
# Train GNNs with different numbers of layers
layer_configs = [1, 2, 3, 4]
results = {}

for num_layers in layer_configs:
    # Create model
    model_temp = GNN(input_dim=input_dim, hidden_dim=hidden_dim, 
                     output_dim=output_dim, num_layers=num_layers).to(device)
    optimizer_temp = torch.optim.Adam(model_temp.parameters(), lr=0.01, weight_decay=5e-4)
    
    # Train
    val_accs_temp = []
    for epoch in range(num_epochs):
        train_epoch(model_temp, X_karate, A_karate, labels, train_mask, optimizer_temp)
        val_acc = evaluate(model_temp, X_karate, A_karate, labels, val_mask)
        val_accs_temp.append(val_acc)
    
    results[num_layers] = val_accs_temp
    print(f"{num_layers} layer(s): Final Val Acc = {val_accs_temp[-1]:.4f}")

Plot validation accuracy for different numbers of layers.

In [ ]:
plt.figure(figsize=(10, 6))

for num_layers, accs in results.items():
    plt.plot(accs, label=f'{num_layers} layer(s)', linewidth=2)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('Effect of Network Depth on GNN Performance', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** 
- **2-3 layers** work well for this small graph
- **Too many layers** can lead to over-smoothing (all nodes become too similar)
- For most graphs, 2-4 layers is a sweet spot

This is different from CNNs where very deep networks (50+ layers) often work better!

## 13. Visualizing Message Passing

Let's visualize how a single node aggregates information from its neighbors through message passing.

In [ ]:
# Pick an interesting node (node 0 - the instructor)
target_node = 0

# Find its neighbors
neighbors = torch.where(A_karate[target_node] > 0)[0].cpu().numpy()

print(f"Node {target_node} has {len(neighbors)} neighbors: {neighbors.tolist()}")
print(f"Node {target_node}'s label: {labels[target_node].item()}")
print(f"Neighbors' labels: {[labels[n].item() for n in neighbors]}")

Visualize the ego network (target node + its neighbors).

In [ ]:
# Create subgraph
ego_nodes = [target_node] + neighbors.tolist()
subgraph = karate.subgraph(ego_nodes)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(subgraph, seed=42)

# Color: target node vs neighbors
node_colors = ['gold' if node == target_node else 'lightblue' for node in subgraph.nodes()]
node_sizes = [1000 if node == target_node else 600 for node in subgraph.nodes()]

nx.draw(subgraph, pos, node_color=node_colors, node_size=node_sizes,
        with_labels=True, font_size=12, font_weight='bold',
        edge_color='gray', width=2, alpha=0.7)

plt.title(f'Node {target_node} and Its Neighbors\n(Gold = target, Blue = neighbors)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

**Message passing in action:** Node 0 (gold) receives messages from all its neighbors (blue) and aggregates them to update its representation.

## 14. Key Takeaways

**Core Intuitions:**

1. **Graph Neural Networks learn on graph-structured data** — social networks, molecules, knowledge graphs, etc.

2. **Message passing is the key idea:**
   - Aggregate neighbor features
   - Transform with learnable weights
   - Repeat across layers

3. **Matrix multiplication `A @ X` elegantly implements message passing** — summing neighbor features in parallel.

4. **Normalization matters** — divide by degree to avoid bias toward high-degree nodes.

5. **Self-loops are important** — nodes need to retain their own information while aggregating from neighbors.

6. **Depth is limited** — 2-4 layers is typical (unlike CNNs). Too many layers cause over-smoothing.

7. **GNNs learn better representations than MLPs** — by incorporating graph structure through neighborhood aggregation.

8. **Semi-supervised learning works well** — GNNs propagate labels through the graph structure, requiring fewer labeled examples.

**When to use GNNs:**
- Social network analysis (community detection, link prediction)
- Molecular property prediction
- Recommendation systems (user-item graphs)
- Knowledge graph reasoning
- Traffic prediction (road networks)
- Any domain where relationships between entities matter!

**Connection to other architectures:**
- **CNNs:** GNNs are like CNNs for irregular grids (graphs)
- **Attention:** Some GNN variants use attention weights for aggregation (Graph Attention Networks)
- **Transformers:** Can be seen as fully-connected GNNs on sequence graphs

## What's Next?

**Extensions to explore:**
- Graph Attention Networks (GAT) — learn which neighbors to attend to
- Graph pooling — hierarchical graph representations
- Edge features — learn on both nodes and edges
- Directed graphs — handle asymmetric relationships
- Temporal graphs — dynamic graphs that change over time
- Graph-level tasks — classify entire graphs (e.g., molecule property prediction)

**Practical libraries:**
- PyTorch Geometric (PyG)
- Deep Graph Library (DGL)
- NetworkX (for graph manipulation)

You now understand the core principles of Graph Neural Networks! The beauty is in the simplicity: aggregate neighbor information, transform, repeat.